# Architecture Comparison of qrisp.qaoa vs new proposed qrisp-optimization module

In this notebook the benefits of the proposed architecutre for a new qrisp optimization modul is compared to the existing qaoa module.

The following UML-Diagram depicts the proposed architecture:

![./optimization.png](./optimization.png)

This notebook focuses on the SOLID Principals and gives an extra example of the "Locality of Behavior" Principal

# Example 1: Solid Principals --- Single Responsibility

In [1]:
import numpy as np
from qrisp import QuantumArray
from qrisp_optimization.utils import QuantumBinary

from qrisp.qaoa import solve_QUBO

In [2]:
Q = np.array([  
        [ 1. , -3. , -2. ,  2. ,  1.5,  0. ,  0. ],
        [ 0. ,  2.5, -2. ,  0. , -1.5,  0.5,  0. ],
        [ 0. ,  0. ,  3. , -2.5, -4. ,  1. ,  0. ],
        [ 0. ,  0. ,  0. ,  1. ,  0. ,  0. ,  2. ],
        [ 0. ,  0. ,  0. ,  0. ,  2. ,  0. ,  0. ],
        [ 0. ,  0. ,  0. ,  0. ,  0. ,  1.5,  0. ],
        [ 0. ,  0. ,  0. ,  0. ,  0. ,  0. ,  1. ]    
    ])

In [3]:
solve_QUBO(Q, depth=1, shots=5000)[:5]

[(np.float64(-2.5),
  OutcomeArray(['1', '1', '1', '0', '1', '0', '0'], dtype=object),
  0.1056),
 (np.float64(-2.0),
  OutcomeArray(['1', '1', '1', '1', '1', '0', '0'], dtype=object),
  0.04),
 (np.float64(-1.5),
  OutcomeArray(['0', '1', '1', '1', '1', '0', '0'], dtype=object),
  0.0492),
 (np.float64(-1.5),
  OutcomeArray(['1', '1', '1', '0', '1', '0', '1'], dtype=object),
  0.0368),
 (np.float64(-0.5),
  OutcomeArray(['1', '1', '1', '0', '0', '0', '0'], dtype=object),
  0.0542)]

The main issue is that the name `solve_QUBO` does not make it clear which method is used to solve the QUBO problem. 
Internally, it relies on QAOA, but the ansatz is effectively fixed: replacing it is practically impossible. 
The processing of the problem formulation is tightly coupled to the specific algorithm used to solve it. 
As a result, this method offers no flexibility, yet it is exposed as part of the public API.

Suggested fix for this issue:

In [4]:
from qrisp_optimization.problems import QUBO_Problem
from qrisp_optimization import QAOA

problem = QUBO_Problem(Q=Q)
algorithm = QAOA(                           
    problem=problem,
    qarg=QuantumArray(
        qtype=QuantumBinary(),
        shape=len(Q)   
    )
)

theta = np.array([0.5]*6)
#theta = algorithm.optimize(theta)
qarg = algorithm.ansatz(theta)
meas = qarg.get_measurement()

This design provides significantly more flexibility:
1. It becomes possible to swap out `QAOA` for a different algorithm (the required changes are described later).
2. Techniques such as warm starts, skipping the internal optimization loop, or externalizing the optimization can be used immediately.
3. It enforces a clear separation of concerns: the algorithm implementation does not need to know where the problem comes from, and the problem definition does not need to know how the algorithm works.
4. It leads to a more structured API. Instead of having standalone functions like `solve_QUBO`, `create_QUBO_cost_operator`, etc., methods are organized around the problem type, for example: `QUBO_Problem(Q=Q).cost_layer(qarg, gamma)`. At the top-level API, only constructs such as `QUBO_Problem`, `MaxCut_Problem`, and algorithms like `QAOA`, `LR_QAOA`, etc. are exposed.

# Example 2: sOlid Principals --- Open Closed Principal

This principle is fundamentally about maintainability: keeping implementations closed to modification, so that introducing new bugs becomes less likely, while still keeping them open for extension.

Consider implementing a new algorithm such as CD‑QAOA from a qrisp user’s perspective (not as a core developer). The requirements of the existing algorithms are no longer sufficient, and you do not have direct access to the codebase to add what you need. With this principle in place, you can still extend the system and introduce your own algorithm, without having to change the existing implementation.

In [5]:
from abc import ABC, abstractmethod
class MyCDQAOA_Requirements(ABC):
    @abstractmethod
    def my_new_required_method(self, someArgument): ...

class MyCD_QUBO_Problem(QUBO_Problem):
    def my_new_required_method(self, someArgument):
        return "what ever this does"


myCD_QUBO_Problem = MyCD_QUBO_Problem(Q=Q)
myCD_QUBO_Problem.cl_cost_function(np.array([1]*7))

4.0

`myCD_QUBO_Problem` gives you access to all methods already implemented in `QUBO_Problem`, so you do not need to reimplement, for example, the classical cost function.

What we explicitly want to avoid is overriding existing behaviour. Doing so would require additional testing and violates both the Open–Closed Principle and the Liskov Substitution Principle. For instance, to implement Linear Ramp QAOA it is preferable to define a new algorithm class from scratch rather than overriding the ansatz of an existing algorithm such as `QAOA`.

# Example 3: soLid Principals --- Liskov Substitution Principle

The `solve_QUBO method` naturally cannot be used to solve Max-Cut instances, so users have to look for a different entry point, even if they want to apply the same underlying algorithm.

By adhering to the Substitution Principle, any problem type that satisfies the algorithm’s interface requirements can be used interchangeably. The same applies to swapping out the algorithm itself: as long as two algorithms share the same requirements (for example, `QAOA` and `LR_QAOA`), you can simply plug the same problem instance into the new algorithm $-$ and you are done.

In [6]:
from qrisp_optimization.problems import MaxCut_Problem

# Example without an actual implementation at this stage
algorithm = QAOA(
    problem=MaxCut_Problem(Q=Q), # this line changed (MaxCut may have a different set of arguments in the constructor though)
    qarg=QuantumArray(
        qtype=QuantumBinary(),
        shape=len(Q)   
    )
)

theta = np.array([0.5]*6)
#theta = algorithm.optimize(theta)
qarg = algorithm.ansatz(theta)
meas = qarg.get_measurement()

This is currently just a dummy class without correct implementation


# Example 4: solId Principals --- Interface Segregation Principle

Only implement the requirements that your algorithm actually needs.

With respect to this principle, the new architecture does not offer direct advantages over the old one, but it is also not worse.

**Example — Solving 3‑SAT with QAOA:**  
In the old architecture, you define the methods for state preparation, classical cost function, and so on.  
In the new architecture, you define a class that implements `QAOA_Requirements` (which consists of exactly the same methods as before).  
The main benefit of this approach is that Python’s type system clearly indicates which methods you need to implement.

In [7]:
from qrisp_optimization import QAOA_Requirements

class ThreeSat_Problem(QAOA_Requirements):
    def state_prep(self, qarg):
        return qarg

try:
    threeSat_Problem = ThreeSat_Problem()
except Exception as e:
    print(e)

Can't instantiate abstract class ThreeSat_Problem without an implementation for abstract methods 'cl_cost_function', 'cost_layer', 'mixer_layer'


As you can see, when the problem is instantiated, we immediately receive an error because we “promised” to implement QAOA_Requirements but failed to do so.

In [8]:
class ThreeSat_Problem(QAOA_Requirements):
    def cl_cost_function(self, x):
        # TODO implement this
        return 0.0

    def state_prep(self, qarg):
        # TODO implement this
        return qarg

    def cost_layer(self, qarg):
        # TODO implement this
        return qarg

    def mixer_layer(self, qarg):
        # TODO implement this (might just be a call to X_Mixer?)
        return qarg

threeSat_Problem = ThreeSat_Problem()

Once all required methods have been implemented, the algorithm’s expectations are fully satisfied and the problem instance can be instantiated without any issues.

# Example 5: soliD Principals --- Dependency Inversion Principle

High‑level modules should depend on abstractions rather than low‑level details.

This is precisely what the new architecture achieves. Algorithms define a set of abstract requirements that problem classes can implement. Whenever a problem type satisfies these requirements, the high‑level algorithm can operate on (and solve) that problem without needing to know any of its concrete implementation details.

# Example 6: Locality of Behavior

If we decide to change the internal representation of graph coloring, for example from a sequential encoding to a one‑hot encoding, all necessary modifications are localized within the GraphColoring_Problem class.

Code outside this class does not need to be aware of the change; as long as the class continues to satisfy the declared requirements, the algorithms will keep working unchanged. This significantly improves maintainability and testability across the entire framework.